run this notebook after running the initialise_db one.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.bronze_restaurant;

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
source_database = "workspace.restaurantdb"
tables_to_move = ["details", "menu", "employees", "orders", "weather"]

We ingest the tables that interest us to the bronze layer while keeping their initial names.

In [0]:
def ingest_to_bronze(source_db, table_name, target_schema="workspace.bronze_restaurant"):
    """
    Ingests data from an existing Databricks table into the Bronze layer 
    with added metadata for auditing.
    """
    print(f"Ingesting {source_db}.{table_name} into {target_schema}.{table_name}.")
    

    df_raw = spark.table(f"{source_db}.{table_name}")
    

    df_bronze = df_raw.withColumn("ingestion_timestamp", current_timestamp()) \
                      .withColumn("source_system", lit(source_db))
    
    df_bronze.write.format("delta") \
              .mode("overwrite") \
              .option("overwriteSchema", "true") \
              .saveAsTable(f"{target_schema}.{table_name}")
    
    print(f"Successfully loaded {table_name}.")

In [0]:
for t in tables_to_move:
    ingest_to_bronze(source_database, t)